In [25]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [26]:
load_dotenv(override=True)
openai = OpenAI()

In [28]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user exists")
else:
    print("Pushover user not found")

Pushover user exists


In [29]:
def push(message):
    payload = {
        "token": pushover_token,
        "user": pushover_user,
        "message": message
    }
    response = requests.post(pushover_url, data=payload)
    print (response)

In [ ]:
#push("hola canalla")

<Response [200]>


In [30]:
def record_user_details (email, name="Not provided", notes ="not provided"):
    push(f"recording interest from {name} with email {email} and notes {notes}")
    return {"recorded":"ok"}

In [31]:
def record_unknown_question (question):
    push(f"recording unknown question: {question}")
    return {"recorded":"ok"}

In [ ]:
def record_job_offer (company="Not provided", job_rol="Not provided", salary="Not provided", conditions="Not provided", remote_working="Not provided"):
    push(f"recording job offer in company: {company} - {job_rol}, {salary}, {conditions}, {remote_working} ")
    return {"recorded":"ok"}

In [32]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string", 
                "description": "The email address of this user"
            },
            "name": {
                "type": "string", 
                "description": "The user's name, if they provided it"
            },
            "notes": {
                "type": "string", 
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [34]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string", 
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [47]:
record_job_offer_json = {
    "name": "record_job_offer",
    "description":"Always use this tool to record any details regarding a job offer from the user",
    "parameters": {
        "type": "object",
        "properties": {
            "company": {
                "type":"string",
                "description":"The company that offers me the job offer"
            },
            "job_rol": {
                "type":"string",
                "description":"The rol of the job offer"
            },
            "salary": {
                "type":"string",
                "description":"Gross annual salary of the job offered"
            },
            "conditions": {
                "type":"string",
                "description":"Any relevant job's conditions"
            },
            "remote_working": {
                "type":"string",
                "description":"Work conditions, If job offer is in remote, hybrid or presential"
            }
        }
    }
}

In [49]:
tools = [{"type": "function", "function": record_user_details_json}, 
        {"type": "function", "function": record_unknown_question_json},
        {"type": "function", "function" : record_job_offer_json}]        

In [50]:
def handle_tool_calls(tool_calls):
    results = []
    print (tool_calls)
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        if tool_name == "record_user_details":
             result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
        elif tool_name == "record_job_offer":
            result = record_job_offer(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})

    return results

In [ ]:
#globals()["record_unknown_question"]("this is a really hard question")

In [42]:
# def handle_tool_calls(tool_calls):
#     results = []
#     for tool_call in tool_calls:
#         tool_name = tool_call.function.name
#         arguments = json.loads(tool_call.function.arguments)
#         print(f"Tool called: {tool_name}", flush=True)
#         tool = globals().get(tool_name)
#         result = tool(**arguments) if tool else {}
#         results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
#     return results

In [37]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Javier Lopez"

In [52]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible\
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. \
If the user propose a job offer, ask him for conditions: ask in the most polite way possible for company, rol, salary, conditions and record it using your record_job_offer tool. \
    "
system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [54]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        # This is the call to the LLM - see that we pass in the tools json
        response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        print (finish_reason)
        # If the LLM wants to call a tool, we do that!
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [55]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


stop
stop
stop
stop
stop
stop
tool_calls
[ChatCompletionMessageFunctionToolCall(id='call_AzJQ3a76zPpOoirwDnUvATMb', function=Function(arguments='{"email":"fj.lopez.abad@gmail.com","notes":"User is proposing a job offer at Equifax for a Business Analyst role in Madrid, hybrid work, with a salary of 50k."}', name='record_user_details'), type='function')]
Tool called: record_user_details
<Response [200]>
stop
